# 2026-06-14 Day 2: Causal Attention And Manual Transformer Core

Goal: understand scaled dot-product attention, causal masking, multi-head shapes, and how the manual and trainable attention paths relate.


## How To Use This Reading Notebook

Use this before touching implementation for the day. The goal is not to memorize the paper. The goal is to extract the model of the system you need for today's code and notes.

Workflow:

1. Read only the assigned sections.
2. Fill the mini-lecture answer in your own words.
3. Open the repo files listed in the roadmap.
4. Run the listed checks or record why they skip.
5. Write a short exit ticket before moving on.


## Assigned Reading

| Source | Exact Target | Why It Matters Today |
| --- | --- | --- |
| [Attention Is All You Need](https://arxiv.org/abs/1706.03762) | Section 3.2 | Defines scaled dot-product attention and multi-head attention. |
| [PyTorch SDPA docs](https://docs.pytorch.org/docs/2.12/generated/torch.nn.functional.scaled_dot_product_attention.html) | Function signature, shapes, `is_causal`, dropout warning | Shows the optimized attention interface you may compare with later. |


## Reading Summary

### Attention Is All You Need, Section 3.2

Scaled dot-product attention computes query-key scores, scales by `sqrt(d_k)`, applies softmax, and uses the result to mix values. Multi-head attention repeats this in smaller subspaces so different heads can learn different lookup patterns.

For a decoder-only LM, the important modification is the causal mask. Position `i` can read positions `j <= i`, but not future positions `j > i`. The mask must happen before softmax so future positions receive probability zero.

### PyTorch `scaled_dot_product_attention` Docs

PyTorch's SDPA is a fused attention API that can apply an attention mask or causal masking. For this project, the key lesson is the expected tensor layout: queries, keys, and values are already projected and arranged with head dimensions before calling the function. The docs also warn that dropout behavior depends on the `dropout_p` argument, so evaluation code must pass zero dropout when needed.

Key takeaway for implementation: SDPA can replace the explicit score-mask-softmax-value sequence later, but you still need to understand the explicit tensor shapes first.


## Diagram Anchor

![Causal mask matrix](../../assets/figures/causal_mask_matrix.svg)

What to notice: every red cell is a future token position and must be blocked before softmax.

![Attention memory scaling](../../assets/figures/attention_memory_scaling.svg)

What to notice: full prefill uses a square score matrix, while one-token cached decode uses one score row over cached keys.


## Repo Connection

Open these files after reading:

| File | What To Find |
| --- | --- |
| `src/moe_transformer_lab/manual/attention.py` | `causal_mask`, `masked_row_softmax`, `ScaledDotProductAttention`, `MultiHeadSelfAttention`. |
| `src/moe_transformer_lab/manual/transformer.py` | How attention is embedded inside the manual decoder block. |
| `src/moe_transformer_lab/trainable/model.py` | Current trainable `CausalSelfAttention`. |
| `tests/test_manual_attention.py` | Expected mask and backward behavior. |


## Shape Summary

| Tensor | Shape | Meaning |
| --- | --- | --- |
| `x` | `(B, T, C)` | residual stream into attention |
| `q`, `k`, `v` before split | `(B, T, C)` | projected full-width tensors |
| `q`, `k`, `v` after split | `(B, H, T, Dh)` | per-head tensors, where `Dh = C / H` |
| `scores` | `(B, H, T, T)` | every target position scores every source position |
| `mask` | `(T, T)` | lower-triangular allowed positions |
| `attn` | `(B, H, T, T)` | row-softmax probabilities |
| output | `(B, T, C)` | heads merged and projected back |


## Mini-Lecture Answer Seed

Causal self-attention lets each token query earlier tokens. The query and key dot product produces a `T by T` matrix because every output position may need to compare against every input position. Causality removes future columns from each row. Splitting into heads reshapes the channel dimension from `C` into `H * Dh`, runs smaller attentions in parallel, then concatenates heads back to `C`.


## Active Recall

1. Why is the mask lower triangular?
2. What does row `i`, column `j` of the score matrix mean?
3. Why do scores scale as `O(T^2)`?
4. What changes if SDPA replaces manual attention?

## Exit Ticket

Write one full attention shape table and one paragraph explaining why masking must happen before softmax.
